## Create plotly plot

In [6]:
!pip install --upgrade plotly nbformat



In [16]:
import pandas as pd
import json
traces = pd.read_csv("rfi_spectrum_traces.csv")
h_lines = pd.read_csv("H_lines_with_clean_scores.csv")
with open("rfi_params.json", "r") as f:
    rfi_params = json.load(f)
rfi_params

{'sk_estimator': {'N': 2343.75,
  'empirical_mode': 1.0407104832483614,
  'd': 0.9608820282838969,
  'Nd': 2252.0672537903833,
  'ref_db': -173},
 'rfi_thresholds': {'SK_low': 0.7913373950013837,
  'SK_high': 1.2418535003811635,
  'gamma_percentile': 0.1,
  'gamma_percentile_upper': 99.9},
 'cutoffs': {'segment_agreement_fraction': 0.5, 'line_clean_threshold': 0.99},
 'line_scoring': {'velocity_width_kms': 100.0, 'n_sigma': 3},
 'resampling': {'velocity_resolution_kms': 50.0, 'n_traces': 10},
 'band': {'chord_min_mhz': 300, 'chord_max_mhz': 1500, 'n_channels': 360000},
 'summary': {'n_segments': 11,
  'cal_cutoff_db': -174,
  'clean_channel_fraction': 0.7629805555555556,
  'n_lines': 116,
  'n_clean_lines': 79,
  'clean_line_fraction': 0.6810344827586207}}

In [18]:
rfi_params["rfi_thresholds"]["SK_low"]

0.7913373950013837

In [12]:
traces

,Frequency (MHz),trace_0,trace_1,trace_2,trace_3,trace_4,trace_5,trace_6,trace_7,trace_8,trace_9,SK_cal,combined_clean
0,300.000000,-172.646576,-172.866211,-172.924393,-172.950195,-172.784409,-173.079941,-172.990128,-172.719589,-172.750427,-172.735809,1.030337,1.0
1,300.050000,-172.767288,-172.878586,-172.950546,-173.003433,-172.967361,-173.001236,-172.869019,-172.769073,-172.992294,-172.703400,0.989719,1.0
2,300.100000,-172.788391,-172.907700,-172.909058,-173.032455,-172.718246,-172.988007,-172.765228,-172.883163,-172.925125,-172.989639,0.976141,1.0
3,300.150000,-172.785156,-172.787079,-172.876404,-172.919708,-172.833755,-173.070709,-172.717087,-173.118210,-172.861511,-172.923538,1.024026,1.0
4,300.200001,-172.783844,-172.928131,-172.787506,-173.244766,-172.866623,-172.794861,-172.915375,-172.929581,-172.965744,-172.700470,0.977881,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9646,1498.813330,-173.232864,-173.325714,-173.319687,-173.279312,-173.395538,-173.478363,-173.183731,-173.125839,-173.293655,-173.141495,0.998282,1.0
9647,1499.063331,-173.165909,-173.476746,-173.298599,-173.213226,-173.248627,-173.352127,-173.228317,-173.060913,-173.260498,-173.076660,0.996180,1.0
9648,1499.313331,-173.165848,-173.330536,-173.132675,-173.430344,-173.164139,-173.215118,-173.124161,-173.281082,-173.393341,-173.303009,0.983391,1.0
9649,1499.563332,-173.000626,-173.094864,-173.283447,-173.165955,-173.195312,-173.287109,-173.193100,-173.220688,-173.270844,-173.239578,0.984084,1.0


In [13]:
h_lines

,Transition,Species,Line,Frequency (MHz),delta_f_MHz_100kms,clean_score,clean
0,H164alpha,H,H164,1477.335105,0.492786,1.0,True
1,H165alpha,H,H165,1450.716777,0.483907,1.0,True
2,H166alpha,H,H166,1424.734102,0.475240,1.0,True
3,H167alpha,H,H167,1399.368217,0.466779,1.0,True
4,H168alpha,H,H168,1374.600927,0.458518,1.0,True
...,...,...,...,...,...,...,...
111,H275alpha,H,H275,314.489820,0.104903,1.0,True
112,H276alpha,H,H276,311.089945,0.103768,1.0,True
113,H277alpha,H,H277,307.738901,0.102651,1.0,True
114,H278alpha,H,H278,304.435814,0.101549,1.0,True


In [51]:
import numpy as np

import plotly.graph_objects as go
from plotly.subplots import make_subplots

freq = traces["Frequency (MHz)"].to_numpy()
trace_cols = [c for c in traces.columns if c.startswith("trace_")]
SK = np.clip(traces["SK_cal"], 0, 3)

is_clean = traces["combined_clean"] == 1
# Per-point "clean"/"RFI" label, fed to hovertemplate via customdata.
status = np.where(is_clean, "clean", "RFI")

# Two stacked panes that share the x-axis: zoom/pan on either one drives both.
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,          # linked frequency axis -> synchronised zoom
    vertical_spacing=0.04,
    row_heights=[0.65, 0.35],
    subplot_titles=("RFI-monitor Power Spectral Density (dBm/Hz)", "Calibrated spectral kurtosis"),
)

# --- top pane: all raw traces in ONE trace (NaN-separated) for speed ---
# 10 separate traces -> 1: far less per-trace overhead in WebGL.
raw = np.clip(traces[trace_cols].to_numpy(), None, -155)   # (N, 10), peaks clipped
nan = np.array([np.nan])
sep_obj = np.array([""], dtype=object)
xs = np.concatenate([np.concatenate([freq, nan]) for _ in trace_cols])
ys = np.concatenate([np.concatenate([raw[:, j], nan]) for j in range(len(trace_cols))])
cd = np.concatenate([np.concatenate([status, sep_obj]) for _ in trace_cols])

fig.add_trace(
    go.Scattergl(
        x=xs, y=ys, mode="markers", name="raw traces",
        marker=dict(size=1, color="blue"), opacity=0.6,
        customdata=cd,
        hovertemplate="Frequency: %{x:.3f} MHz<br>"
                      "Power: %{y:.2f} dBm/Hz<br>"
                      "Status: %{customdata}<extra></extra>",
    ),
    row=1, col=1,
)

# --- bottom pane: calibrated SK in ONE trace, coloured per-point ---
# marker.color takes a per-point array; customdata packs [true SK, status label]
# (object dtype keeps SK numeric so %{...:.3f} formats and the label stays text).
sk_colors = np.where(is_clean, "green", "red")
sk_customdata = np.empty((len(traces), 2), dtype=object)
sk_customdata[:, 0] = traces["SK_cal"].to_numpy()   # unclipped SK value
sk_customdata[:, 1] = status                        # "clean" / "RFI"

fig.add_trace(
    go.Scattergl(
        x=freq, y=SK, mode="markers", name="SK_cal",
        marker=dict(size=3, color=sk_colors),
        customdata=sk_customdata,
        hovertemplate="Frequency: %{x:.3f} MHz<br>"
                      "SK: %{customdata[0]:.3f}<br>"
                      "Status: %{customdata[1]}<extra></extra>",
    ),
    row=2, col=1,
)
fig.add_hline(y=rfi_params["rfi_thresholds"]["SK_low"], line=dict(color="gray", dash="dash", width=1), row=2, col=1)
fig.add_hline(y=rfi_params["rfi_thresholds"]["SK_high"], line=dict(color="gray", dash="dash", width=1), row=2, col=1)

# --- H-line markers as ONE line trace per pane (NaN-separated), not 232 shapes ---
# Layout shapes re-render on every zoom/pan; trace data lives in WebGL and doesn't.
hlf = h_lines["Frequency (MHz)"].to_numpy()
top_lo, top_hi = np.nanmin(raw), -155         # y-extent of the raw pane
vx = np.repeat(hlf, 3)
vline_kw = dict(mode="lines", line=dict(color="black", dash="dash", width=0.5),
                opacity=0.3, hoverinfo="skip", showlegend=False, name="H lines")
fig.add_trace(go.Scattergl(x=vx, y=np.tile([top_lo, top_hi, np.nan], len(hlf)), **vline_kw),
              row=1, col=1)
fig.add_trace(go.Scattergl(x=vx, y=np.tile([0, 3, np.nan], len(hlf)), **vline_kw),
              row=2, col=1)

fig.update_yaxes(title_text="Power (dBm/Hz)", row=1, col=1)
fig.update_yaxes(title_text="Calibrated Spectral Kurtosis", row=2, col=1)
fig.update_xaxes(title_text="Frequency (MHz)", row=2, col=1)

# --- dropdown: pick an H line and zoom the shared x-axis to it ---
WINDOW_MHZ = 2.0   # half-width of the zoom window around the selected line
hl = h_lines.sort_values("Frequency (MHz)", ascending=False)

def zoom_args(lo, hi):
    # both x-axes are shared, but set each so the relayout is unambiguous
    return [{"xaxis.range": [lo, hi], "xaxis2.range": [lo, hi]}]

band_lo = rfi_params["band"]["chord_min_mhz"]
band_hi = rfi_params["band"]["chord_max_mhz"]
buttons = [dict(label="Full band", method="relayout",
                args=zoom_args(band_lo, band_hi))]
for _, r in hl.iterrows():
    f0 = r["Frequency (MHz)"]
    line_status = "clean" if r["clean"] else "RFI"
    buttons.append(dict(
        label=f'{f0:.2f} MHz ({line_status})',
        method="relayout",
        args=zoom_args(f0 - WINDOW_MHZ, f0 + WINDOW_MHZ),
    ))

fig.update_layout(
    height=750,
    title="CHORD RFI-monitor: Measured Power, Spectral Kurtosis RFI labels, and H-line Visibility",
    hovermode="x unified",
    legend=dict(orientation="v", x=1.02, xanchor="left", y=0.5),
    margin=dict(t=110, b=50, r=140),
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        showactive=True,
        x=1.02, xanchor="left", y=1.0, yanchor="top",
        pad=dict(t=2, b=2),
    )],
    annotations=list(fig.layout.annotations) + [dict(
        text="Zoom to line:", x=1.02, xref="paper", xanchor="left",
        y=1.02, yref="paper", yanchor="bottom", showarrow=False,
        font=dict(size=15),
    )],
)

# Drag = box-zoom (both panes zoom together on x). Use the modebar for pan,
# double-click to reset, or pick a line from the dropdown to zoom to it.
fig.show()


In [52]:
import os

# --- write the figure + description to a static GitHub Pages page (docs/) ---
DOCS = os.path.join("..", "..", "docs")
os.makedirs(DOCS, exist_ok=True)

band_lo = rfi_params["band"]["chord_min_mhz"]
band_hi = rfi_params["band"]["chord_max_mhz"]

# JS run after the plot is built: zoom-in makes the markers larger and the
# H-line verticals thicker/darker. z = 0 at full band and grows (log2) as the
# visible span shrinks.
resize_js = f"""
var gd = document.getElementById('rfi-plot');
var FULL = {band_hi - band_lo};
function resizeMarkers() {{
    var ax = gd.layout.xaxis;
    var r = (ax && ax.range) ? ax.range : null;
    var span = r ? Math.abs(r[1] - r[0]) : FULL;
    var z = Math.max(0, Math.log2(FULL / span));
    Plotly.restyle(gd, {{'marker.size': Math.min(6, 1 + z)}}, [0]);   // raw traces
    Plotly.restyle(gd, {{'marker.size': Math.min(6, 3 + z)}}, [1]);   // SK markers
    // H-line verticals (traces 2 = top pane, 3 = bottom pane): thicker + darker
    Plotly.restyle(gd, {{
        'line.width': Math.min(2.5, 0.5 + 0.4 * z),
        'opacity':    Math.min(1.0, 0.3 + 0.15 * z)
    }}, [2, 3]);
}}
gd.on('plotly_relayout', resizeMarkers);
"""

# The plot as an inline HTML fragment (a <div>); plotly.js pulled from the CDN
# to keep the page small. responsive=True lets it fill its grid column.
plot_div = fig.to_html(
    full_html=False,
    include_plotlyjs="cdn",
    div_id="rfi-plot",
    config={"responsive": True},
    post_script=resize_js,
)

th = rfi_params["rfi_thresholds"]
cut = rfi_params["cutoffs"]
summ = rfi_params["summary"]

# values pulled out to keep the f-string readable
res = rfi_params["resampling"]["velocity_resolution_kms"]
vwidth = rfi_params["line_scoring"]["velocity_width_kms"]
nseg = summ["n_segments"]
agree = cut["segment_agreement_fraction"] * 100
cleanfrac = summ["clean_channel_fraction"] * 100
lo, hi = th["SK_low"], th["SK_high"]
plo, phi = th["gamma_percentile"], th["gamma_percentile_upper"]
thr = cut["line_clean_threshold"]
nclean, nlines = summ["n_clean_lines"], summ["n_lines"]

description = f"""
<h2>Instructions</h2>
<p>Pick a line from the <em>Zoom&nbsp;to&nbsp;line</em> dropdown to jump to a given
H&thinsp;n&alpha; line, or drag a box on either panel to zoom into a region. The two panels
share the frequency axis, and hovering reports the frequency, value and clean/RFI status.</p>

<h2>Description</h2>
<p><strong>Top panel</strong> &mdash; 10 power samples from the RFI monitor for each frequency
bin, resampled to {res:.0f}&nbsp;km/s resolution.</p>
<p><strong>Bottom panel</strong> &mdash; the calibrated spectral kurtosis (SK) per channel,
<span style="color:green">green</span> where the channel is clean and
<span style="color:red">red</span> where it is flagged as RFI. Dashed verticals mark the
H&thinsp;n&alpha; lines.</p>

<h2>Methods</h2>
<h3>Segments</h3>
<p>The monitor records {nseg} usable time segments between calibration cycles. All statistics
are computed independently per segment and combined at the end.</p>

<h3>Spectral kurtosis</h3>
<p>Within a segment, each channel accumulates the power sums \\(S_1=\\sum P\\) and
\\(S_2=\\sum P^2\\) over its \\(M\\) time integrations. The generalised SK estimator is</p>
$$\\widehat{{SK}} = \\frac{{Nd\\,M + 1}}{{M-1}}\\left(\\frac{{M\\,S_2}}{{S_1^2}} - 1\\right),$$
<p>where \\(Nd\\) is the calibrated shape parameter. RFI-free data sits at
\\(\\widehat{{SK}}\\approx 1\\).</p>

<h3>Clean / RFI thresholds</h3>
<p>A channel is flagged as RFI when its SK falls outside the central
{plo}%&ndash;{phi}% range of an RFI-free SK distribution, simulated from
\\(\\mathrm{{Gamma}}(Nd,1)\\) draws:</p>
$$SK_{{{plo}\\%}} &lt; \\widehat{{SK}} &lt; SK_{{{phi}\\%}},
\\qquad [{lo:.3f},\\ {hi:.3f}].$$
<p>Segments are combined by majority vote: a channel is clean when more than {agree:.0f}% of
segments agree, leaving <strong>{cleanfrac:.1f}%</strong> of channels clean.</p>

<h3>Per-line visibility score</h3>
<p>Each H&thinsp;n&alpha; line is scored by the Gaussian-weighted fraction of clean bins in a
{vwidth:.0f}&nbsp;km/s window (\\(\\sigma = f_0\\,v/c\\)):</p>
$$\\mathrm{{score}}(f_0) = \\frac{{\\sum_i w_i\\,\\mathrm{{clean}}_i}}{{\\sum_i w_i}},
\\quad w_i = e^{{-\\tfrac{{1}}{{2}}\\left(\\tfrac{{f_i - f_0}}{{\\sigma}}\\right)^2}}.$$
<p>A line is usable when its score \\(\\ge {thr}\\):
<strong>{nclean} / {nlines}</strong> lines qualify.</p>
"""

page = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>CHORD RFI spectrum &amp; H-line visibility</title>
  <link rel="stylesheet" href="assets/style.css">
  <script>
    window.MathJax = {{ tex: {{ inlineMath: [['\\\\(', '\\\\)']], displayMath: [['$$', '$$']] }} }};
  </script>
  <script src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js" async></script>
</head>
<body>
  <header>
    <h1>CHORD RFI-monitor spectrum &amp; H-line visibility</h1>
    <p>Interactive assessment of hydrogen recombination line visibility against measured RFI.</p>
  </header>
  <div class="layout">
    <div class="plot">{plot_div}</div>
    <aside class="desc">{description}</aside>
  </div>
</body>
</html>
"""

out = os.path.join(DOCS, "index.html")
with open(out, "w") as f:
    f.write(page)
print(f"Wrote {out}")


Wrote ../../docs/index.html
